In [ ]:
# ==============================
# Install dependencies
# ==============================
!pip install ipywidgets xgboost lightgbm catboost scikit-learn pandas openpyxl

# ==============================
# Import libraries
# ==============================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              AdaBoostRegressor, ExtraTreesRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# ==============================
# Load dataset
# ==============================
file_path = "/content/RPG.xlsx"  # Change if needed
data = pd.read_excel(file_path)

X = data.drop(['CS', 'Slump', 'CO2 footprint'], axis=1)
y_cs = data['CS']
y_slump = data['Slump']
y_co2 = data['CO2 footprint']
feature_cols = X.columns.tolist()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_cs_train, y_cs_test = train_test_split(X_scaled, y_cs, test_size=0.2, random_state=42)
_, _, y_slump_train, y_slump_test = train_test_split(X_scaled, y_slump, test_size=0.2, random_state=42)
_, _, y_co2_train, y_co2_test = train_test_split(X_scaled, y_co2, test_size=0.2, random_state=42)

# ==============================
# Model definitions
# ==============================
def get_base_models():
    return {
        'XGBoost': XGBRegressor(n_estimators=2000, learning_rate=0.001, max_depth=10,
                                min_child_weight=3, subsample=0.8, random_state=42),
        'AdaBoost': AdaBoostRegressor(n_estimators=2500, learning_rate=0.01, random_state=42),
        'RF (FT)': RandomForestRegressor(min_samples_leaf=12, min_samples_split=6,
                                         max_depth=10, n_estimators=1000, random_state=42),
        'KNR': KNeighborsRegressor(n_neighbors=10, weights='distance', leaf_size=20),
        'BR (FT)': BayesianRidge(max_iter=1000),
        'ETR': ExtraTreesRegressor(n_estimators=1300, min_samples_leaf=3,
                                   min_samples_split=5, random_state=0),
        'GBR': GradientBoostingRegressor(n_estimators=1500, learning_rate=0.001,
                                         min_samples_leaf=3, min_samples_split=5,
                                         max_depth=10, random_state=42),
        'CatBoost': CatBoostRegressor(n_estimators=2000, learning_rate=0.002,
                                      max_depth=12, min_data_in_leaf=3,
                                      verbose=0, random_state=42),
        'ERTR': ExtraTreesRegressor(n_estimators=1000, min_samples_leaf=8,
                                    min_samples_split=6, random_state=0),
        'RF': RandomForestRegressor(min_samples_leaf=15, min_samples_split=8,
                                    max_depth=10, n_estimators=1000, random_state=42),
        'BR': BayesianRidge(max_iter=1200),
        'LGBM': LGBMRegressor(n_estimators=2500, learning_rate=0.01,
                              max_depth=10, num_leaves=20,
                              boosting_type='gbdt', random_state=42)
    }

def create_ann_model():
    return MLPRegressor(
        hidden_layer_sizes=(32, 32, 16),
        activation='relu',
        solver='adam',
        learning_rate='invscaling',
        max_iter=1500,
        random_state=42
    )

class StackedModel:
    def __init__(self, base_models, meta_model):
        self.base_models = base_models
        self.meta_model = meta_model

    def fit(self, X, y):
        base_predictions = np.column_stack([model.predict(X) for model in self.base_models.values()])
        self.meta_model.fit(base_predictions, y)
        return self

    def predict(self, X):
        base_predictions = np.column_stack([model.predict(X) for model in self.base_models.values()])
        return self.meta_model.predict(base_predictions)

def train_target_model(X_train, y_train):
    base_models = get_base_models()
    base_models['ANN'] = create_ann_model()
    meta_model = RandomForestRegressor(n_estimators=2000, random_state=42)

    trained_models = {}
    for name, model in base_models.items():
        model.fit(X_train, y_train)
        trained_models[name] = model

    stacked_model = StackedModel(trained_models, meta_model)
    stacked_model.fit(X_train, y_train)
    return stacked_model

# ==============================
# Train models
# ==============================
print("Training models...")
cs_model = train_target_model(X_train, y_cs_train)
slump_model = train_target_model(X_train, y_slump_train)
co2_model = train_target_model(X_train, y_co2_train)
print("All models trained successfully!")

# ==============================
# NEW USER INTERFACE
# ==============================

# --- Title and Authors (merged + centered) ---
title_authors_box = widgets.HTML(
    value="""
    <div style="
        background-color:#4CAF50;
        color:white;
        padding:15px;
        border-radius:10px;
        text-align:center;">
        <h2 style="margin:5px;color:black";>Transfer-Learned Stacked Models for Assessing Mechanical and Environmental Properties of Glass Powder UHPC</h2>
        <p style="margin:5px; font-size:16px; color:black";><b>Developed by:</b> Abba Bashir, AIB Farouk, Sani I. Abba</p>
    </div>
    """
)

# --- Input Section ---
feature_cols = ["Cement", "SS", "Silica Fume (SF)", "Glass Powder (GP)", "SP",
                "w/b", "Steel Fiber", "Mixing Time", "Curing Temp (°C)"]

input_widgets = {
    "Cement": widgets.FloatText(description="Cement"),
    "SS": widgets.FloatText(description="SS"),
    "Silica Fume (SF)": widgets.FloatText(description="Silica Fume"),
    "Glass Powder (GP)": widgets.FloatText(description="Glass Powder"),
    "SP": widgets.FloatText(description="SP (%)"),
    "w/b": widgets.FloatText(description="w/b ratio"),
    "Steel Fiber": widgets.FloatText(description="Steel Fiber"),
    "Mixing Time": widgets.FloatText(description="Mixing Time"),
    "Curing Temp (°C)": widgets.FloatText(description="Curing Temp")
}

# Centered label
input_label = widgets.HTML(
    "<div style='background-color:#4CAF50; text-align:center; font-weight:bold; font-size:16px;'>Input Features</div>"
)

# Group inputs
inputs_box = widgets.VBox(list(input_widgets.values()))

# Container with styling
inputs_container = widgets.VBox(
    [input_label, inputs_box],
    layout=widgets.Layout(
        border="5px solid #4CAF50",
        padding="15px",
        margin="15px",
        background_color="#90EE90",
        border_radius="12px",
        width="100%"
    )
)

# --- Output Section ---
output_label = widgets.HTML(
    "<div style='background-color:#4CAF50; text-align:center; font-weight:bold; font-size:16px;'>Prediction Results</div>"
)

cs_out = widgets.HTML("<b>Compressive Strength (CS):</b> -")
slump_out = widgets.HTML("<b>Slump:</b> -")
co2_out = widgets.HTML("<b>CO₂ Emission:</b> -")

output_box = widgets.VBox([cs_out, slump_out, co2_out])

outputs_container = widgets.VBox(
    [output_label, output_box],
    layout=widgets.Layout(
        border="5px solid #4CAF50",
        padding="15px",
        margin="15px",
        background_color="#90EE90",
        border_radius="12px",
        width="100%"
    )
)

# --- Buttons ---
predict_button = widgets.Button(
    description="Predict",
    button_style="success",
    layout=widgets.Layout(width="150px")
)

reset_button = widgets.Button(
    description="Reset",
    button_style="danger",
    layout=widgets.Layout(width="150px")
)

buttons_box = widgets.HBox(
    [predict_button, reset_button],
    layout=widgets.Layout(justify_content="center")
)

# --- Button Functions ---
def on_predict_clicked(b):
    try:
        # Collect user inputs
        input_values = [input_widgets[feature].value for feature in feature_cols]
        input_array = np.array(input_values).reshape(1, -1)

        # Scale inputs
        input_scaled = scaler.transform(input_array)

        # Predictions
        cs_pred = cs_model.predict(input_scaled)[0]
        slump_pred = slump_model.predict(input_scaled)[0]
        co2_pred = co2_model.predict(input_scaled)[0]

        # Display predictions
        cs_out.value = f"<b>Compressive Strength (CS):</b> {cs_pred:.2f} MPa"
        slump_out.value = f"<b>Slump:</b> {slump_pred:.2f} mm"
        co2_out.value = f"<b>CO₂ Emission:</b> {co2_pred:.2f} kg/m³"

    except Exception as e:
        cs_out.value = f"<b>Error:</b> {str(e)}"
        slump_out.value = "-"
        co2_out.value = "-"

def on_reset_clicked(b):
    for widget in input_widgets.values():
        widget.value = 0.0
    cs_out.value = "<b>Compressive Strength (CS):</b> -"
    slump_out.value = "<b>Slump:</b> -"
    co2_out.value = "<b>CO₂ Emission:</b> -"

predict_button.on_click(on_predict_clicked)
reset_button.on_click(on_reset_clicked)

# --- Grid Layout (Inputs | Outputs) ---
grid_layout = widgets.HBox([inputs_container, outputs_container])

# --- Final Layout ---
app_layout = widgets.VBox([
    title_authors_box,
    widgets.HTML("<hr style='border:10px solid #4CAF50; margin:20px 0;'>"),
    grid_layout,
    buttons_box
])

display(app_layout)


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit